# Build External Data Pipeline

Use this notebook to test the first five external-data steps interactively before turning them into production ingestion jobs. The notebook is free-first and designed around official filings, macro data, broad news discovery, and optional market-data enrichment.


## 0. Setup

- `SEC` = **U.S. Securities and Exchange Commission**. We use SEC data for company master data, filing metadata, company facts, and filing text.
- `FRED` = **Federal Reserve Economic Data** from the Federal Reserve Bank of St. Louis. We use it for macro series like rates, inflation, and unemployment.
- `GDELT` = **Global Database of Events, Language, and Tone**. We use it for broad news and narrative discovery.
- `FMP` = **Financial Modeling Prep**. It is optional and only runs if `FMP_API_KEY` is set in the environment.

### Helpful Short Forms

- `CIK` = **Central Index Key**, the SEC company identifier.
- `API` = **Application Programming Interface**, which is how we request data programmatically.
- `10-K` = annual filing.
- `10-Q` = quarterly filing.
- `8-K` = current event filing for important company updates.
- `MD&A` = **Management’s Discussion and Analysis**, where management explains company performance and risks.
- `Vector DB` = **Vector Database**, used for storing searchable text embeddings rather than numeric market tables.


In [ ]:
from __future__ import annotations
import gzip
import json
import os
import re
from pathlib import Path
from urllib.parse import quote
from urllib.request import Request, urlopen
import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'data_pipeline' else Path.cwd().resolve()
DATA_DIR = ROOT / 'data'
EXTERNAL_DIR = DATA_DIR / 'external'
INTERIM_DIR = DATA_DIR / 'interim'
RAW_DIR = DATA_DIR / 'raw'
load_dotenv(ROOT / '.env')
for path in [EXTERNAL_DIR / 'sec', EXTERNAL_DIR / 'fred', EXTERNAL_DIR / 'gdelt', EXTERNAL_DIR / 'fmp', INTERIM_DIR / 'vector_preview']:
    path.mkdir(parents=True, exist_ok=True)

SEC_HEADERS = {
    'User-Agent': 'portfolio-analyzer research contact@example.com',
    'Accept-Encoding': 'gzip, deflate',
}

def read_response_bytes(response) -> bytes:
    payload = response.read()
    encoding = (response.headers.get('Content-Encoding') or '').lower()
    if 'gzip' in encoding:
        payload = gzip.decompress(payload)
    return payload

def fetch_json(url: str, headers: dict[str, str] | None = None) -> dict | list:
    request = Request(url, headers=headers or {})
    with urlopen(request) as response:
        return json.loads(read_response_bytes(response).decode('utf-8'))

def fetch_text(url: str, headers: dict[str, str] | None = None) -> str:
    request = Request(url, headers=headers or {})
    with urlopen(request) as response:
        return read_response_bytes(response).decode('utf-8', errors='ignore')

def simple_chunk(text: str, chunk_size: int = 1400) -> list[str]:
    clean = re.sub(r'\s+', ' ', text).strip()
    return [clean[i:i + chunk_size] for i in range(0, len(clean), chunk_size) if clean[i:i + chunk_size].strip()]

def html_to_text(html: str) -> str:
    no_script = re.sub(r'<(script|style)[^>]*>.*?</\\1>', ' ', html, flags=re.I | re.S)
    no_tags = re.sub(r'<[^>]+>', ' ', no_script)
    return re.sub(r'\s+', ' ', no_tags).strip()

fake_csv = RAW_DIR / 'fake_mantis_invest.csv'
transactions = pd.read_csv(fake_csv) if fake_csv.exists() else pd.DataFrame()
tickers = sorted({str(t).strip() for t in transactions.get('Instrument', pd.Series(dtype='object')).dropna().tolist() if str(t).strip()})
tickers[:10], len(tickers)

## 1. Master Data

Build the ticker universe, map it to SEC CIKs, and create the company master table. This becomes the base join key for the rest of the pipeline.


In [ ]:
company_master.head()

In [ ]:
company_tickers_url = 'https://www.sec.gov/files/company_tickers.json'
company_tickers = fetch_json(company_tickers_url, headers=SEC_HEADERS)
company_master = pd.DataFrame(company_tickers).T.rename(columns={'ticker': 'ticker', 'title': 'company_name', 'cik_str': 'cik'})
company_master['ticker'] = company_master['ticker'].astype(str).str.upper()
company_master['cik'] = company_master['cik'].astype(str).str.zfill(10)
if tickers:
    company_master = company_master[company_master['ticker'].isin(tickers)].copy()
company_master.to_csv(EXTERNAL_DIR / 'sec' / 'company_master.csv', index=False)
company_master.head(10)

## 2. Structured Core

Start with macro series from FRED and optionally enrich ticker metadata with FMP if an API key is available.


In [ ]:
FRED_API_KEY = os.getenv('FRED_API_KEY', '')
fred_series = {
    'FEDFUNDS': 'Fed Funds Rate',
    'CPIAUCSL': 'CPI',
    'UNRATE': 'Unemployment Rate',
    'DGS10': '10Y Treasury',
    'DGS2': '2Y Treasury',
}
fred_frames = []
for series_id, title in fred_series.items():
    fred_url = (
        'https://api.stlouisfed.org/fred/series/observations'
        f'?series_id={series_id}&api_key={FRED_API_KEY}&file_type=json'
    )
    payload = fetch_json(fred_url)
    frame = pd.DataFrame(payload.get('observations', []))[['date', 'value']]
    frame['series_id'] = series_id
    frame['title'] = title
    fred_frames.append(frame)
fred_df = pd.concat(fred_frames, ignore_index=True) if fred_frames else pd.DataFrame()
fred_df.to_csv(EXTERNAL_DIR / 'fred' / 'macro_series.csv', index=False)
fred_df.tail(10)

In [ ]:
FMP_API_KEY = os.getenv('FMP_API_KEY', '')
if FMP_API_KEY and not company_master.empty:
    sample_tickers = company_master['ticker'].head(5).tolist()
    profile_frames = []
    for symbol in sample_tickers:
        fmp_url = f'https://financialmodelingprep.com/stable/profile?symbol={symbol}&apikey={FMP_API_KEY}'
        profile_payload = fetch_json(fmp_url)
        profile_frames.append(pd.DataFrame(profile_payload if isinstance(profile_payload, list) else [profile_payload]))
    fmp_profiles = pd.concat(profile_frames, ignore_index=True) if profile_frames else pd.DataFrame()
    fmp_profiles.to_csv(EXTERNAL_DIR / 'fmp' / 'company_profiles_preview.csv', index=False)
    display_columns = [col for col in ['symbol', 'companyName', 'sector', 'industry', 'marketCap'] if col in fmp_profiles.columns]
display(fmp_profiles[display_columns].head())
else:
    print('FMP_API_KEY not set. Skip this cell for now or add a free-tier key to test profile enrichment.')

## 3. SEC Filing Metadata + Company Facts

Pull official filing metadata and company facts for one sample ticker so you can validate the SEC ingestion path before scaling it.


In [ ]:
sample_ticker = company_master['ticker'].iloc[0] if not company_master.empty else 'AAPL'
sample_cik = company_master.loc[company_master['ticker'].eq(sample_ticker), 'cik'].iloc[0] if not company_master.empty else '0000320193'
submissions_url = f'https://data.sec.gov/submissions/CIK{sample_cik}.json'
submissions = fetch_json(submissions_url, headers=SEC_HEADERS)
recent = pd.DataFrame(submissions.get('filings', {}).get('recent', {}))
recent = recent[['accessionNumber', 'filingDate', 'form', 'primaryDocument']].head(20)
recent.to_csv(EXTERNAL_DIR / 'sec' / f'{sample_ticker.lower()}_filings_preview.csv', index=False)
display(recent.head(10))
facts_url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{sample_cik}.json'
company_facts = fetch_json(facts_url, headers=SEC_HEADERS)
us_gaap = company_facts.get('facts', {}).get('us-gaap', {})
fact_names = list(us_gaap.keys())[:15]
pd.DataFrame({'ticker': [sample_ticker] * len(fact_names), 'fact_tag': fact_names})

## 4. Text Pipeline / Vector Preview

Take one recent 10-K or 10-Q, fetch the filing text, clean it, and create vector-ready chunks with metadata. This is a preview of what will later go into the vector database.


In [ ]:
candidate_forms = recent[recent['form'].isin(['10-K', '10-Q'])].copy()
if candidate_forms.empty:
    raise ValueError(f'No recent 10-K/10-Q found for {sample_ticker}')
filing_row = candidate_forms.iloc[0]
accession = filing_row['accessionNumber'].replace('-', '')
primary_doc = filing_row['primaryDocument']
filing_url = f'https://www.sec.gov/Archives/edgar/data/{int(sample_cik)}/{accession}/{primary_doc}'
filing_html = fetch_text(filing_url, headers=SEC_HEADERS)
filing_text = html_to_text(filing_html)
chunks = simple_chunk(filing_text, chunk_size=1800)
chunk_preview = pd.DataFrame({
    'ticker': sample_ticker,
    'cik': sample_cik,
    'form_type': filing_row['form'],
    'filing_date': filing_row['filingDate'],
    'chunk_id': [f'{sample_ticker}-{i:03d}' for i in range(len(chunks[:5]))],
    'text_preview': chunks[:5],
})
chunk_preview.to_csv(INTERIM_DIR / 'vector_preview' / f'{sample_ticker.lower()}_sec_chunks_preview.csv', index=False)
chunk_preview

## 5. News / Narrative + Derived Feature Preview

Use GDELT for a quick free narrative pass, then create a tiny derived feature snapshot so you can sanity-check how the structured and text layers will come together later.


In [ ]:
query = quote(sample_ticker)
gdelt_url = (
    'https://api.gdeltproject.org/api/v2/doc/doc'
    f'?query={query}&mode=artlist&maxrecords=10&format=json'
)
gdelt_payload = fetch_json(gdelt_url)
articles = pd.DataFrame(gdelt_payload.get('articles', []))
articles.to_csv(EXTERNAL_DIR / 'gdelt' / f'{sample_ticker.lower()}_news_preview.csv', index=False)
display(articles[['title', 'domain', 'seendate', 'url']].head(10) if not articles.empty else pd.DataFrame())
derived_snapshot = pd.DataFrame([{
    'ticker': sample_ticker,
    'recent_filings_count': len(recent),
    'vector_chunk_count_preview': len(chunks),
    'news_article_count_preview': len(articles),
    'macro_series_tracked': len(fred_series),
}])
derived_snapshot.to_csv(INTERIM_DIR / 'derived_feature_preview.csv', index=False)
derived_snapshot